# FAME Database Inspection & Auditing Tools
This notebook provides a suite of tools to quickly audit, sample, and query the compiled FAME DuckDB database.

It uses `Ibis` to push computations directly to DuckDB, meaning queries execute in milliseconds and use almost zero Python memory.

In [1]:
from pathlib import Path
from dataclasses import dataclass
import os
import sys
from dotenv import load_dotenv

# Add .env's PYTHONPATH to sys.path
load_dotenv(override=True)
PYTHONPATH = os.getenv("PYTHONPATH")
if PYTHONPATH is not None:
    if PYTHONPATH not in sys.path:
        print(f"Adding {PYTHONPATH} to sys.path")
        sys.path.append(PYTHONPATH)

# Define the structure clearly
@dataclass
class Dirs:
    data_dir: Path = None       # type: ignore
    output_dir: Path = None     # type: ignore
    input_dir: Path = None      # type: ignore
dirs = Dirs()

try:
    from utils.f_0_dirs import get_data_dirs
    dirs = get_data_dirs()
    # raise ImportError("Testing ImportError for demonstration purposes") 

# For easy access purposes, if this script is run directly in a folder with the databases
# and without the dir import modules, then just set it to current_dir
except ImportError as e:
    print(f"ImportError: {e}")
    try:
        current_dir = Path(__file__).parent
    except NameError:
        current_dir = Path.cwd()
    dirs = Dirs(output_dir=current_dir, data_dir=current_dir, input_dir=current_dir)

for attr in dir(dirs):
    if attr.startswith('_'):
        continue
    if callable(getattr(dirs, attr)):
        continue
    print(f"{attr}: {getattr(dirs, attr)}")

data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\2_clean_FAME_data
db_path: C:\Users\lazyst\Files\ucl\Dissertation\build\output\fame_data.duckdb
input_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\input
output_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\output
raw_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\1_FAME_raw_data\2025.02
root_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean
root_dir: C:\Users\lazyst\Files\ucl\Dissertation
tmp_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\tmp
work_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\src


In [2]:
import pandas as pd
import ibis
import ibis.selectors as s

# 1. Setup paths
db_path = dirs.output_dir / "fame_data.duckdb"

# 2. Connect to the database
con = ibis.duckdb.connect(str(db_path))

# 3. Configure Pandas display for easier reading
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 50)

print(f"✅ Successfully connected to: {db_path}")

✅ Successfully connected to: C:\Users\lazyst\Files\ucl\Dissertation\build\output\fame_data.duckdb


## 1. High-Level Database Overview
Quickly check which tables exist in the database and their total row counts.

In [3]:
tables = con.list_tables()
print("📊 Database Tables Overview:\n" + "-"*30)

for table_name in tables:
    t = con.table(table_name)
    # Fast row counting pushed down to DuckDB
    row_count = t.count().execute()
    col_count = len(t.columns)
    print(f"[{table_name}]")
    print(f"   Rows: {row_count:,} | Columns: {col_count}")
print("-" * 30)

📊 Database Tables Overview:
------------------------------
[fame_derived]
   Rows: 9,283,479 | Columns: 7
[fame_derived_clean]
   Rows: 8,771,336 | Columns: 7
[fame_fixed]
   Rows: 9,795,622 | Columns: 29
[fame_fixed_clean]
   Rows: 9,283,479 | Columns: 29
[fame_yearly]
   Rows: 48,764,719 | Columns: 37
[fame_yearly_clean]
   Rows: 45,609,753 | Columns: 37
[lars_fixed]
   Rows: 452,638 | Columns: 26
[lars_yearly]
   Rows: 2,378,089 | Columns: 35
[working_yearly_kp]
   Rows: 40,846,539 | Columns: 37
------------------------------


In [4]:
import ibis

out_file = dirs.output_dir / "duckdb_tables.md"
con = ibis.duckdb.connect(str(dirs.output_dir / "fame_data.duckdb"))
interesting_tables = ["fame_fixed", "fame_derived", "fame_yearly", "lars_fixed", "lars_yearly"]
existing_tables = con.list_tables()
intersection = set(interesting_tables).intersection(existing_tables)

with open(out_file, "w") as f:
    f.write("# Tables in DuckDB database\n\n")
    for table in intersection:
        f.write(f"## {table}\n\n")
        f.write(f"### Number of rows: {con.table(table).count().execute():,}\n\n")
        f.write(f"### Schema:\n\n```\n{con.table(table).schema()}\n```\n\n")
        f.write(f"### Head of table:\n\n```\n{con.table(table).head().execute()}\n```\n\n")
print(f"✅ Successfully listed tables and their heads in: {out_file}")

✅ Successfully listed tables and their heads in: C:\Users\lazyst\Files\ucl\Dissertation\build\output\duckdb_tables.md


## 2. Fast Random Sampling (Reservoir Sampling)
Extract a random subset of rows for visual inspection. We use DuckDB's native reservoir sampling (`USING SAMPLE X ROWS`) so it returns instantly, even on tables with millions of rows, without doing a full table scan.

In [5]:
target_table = "fame_yearly"
sample_frac = 0.001  # Sample fraction for random sampling

# Native Ibis sampling using positional row count and seed parameter
table = con.table(target_table)
row_count = table.count().execute()
df_sample = table.sample(sample_frac, seed=12345).execute()

print(f"🎲 Random sample of {row_count:,} rows from '{target_table}':")
display(df_sample)

🎲 Random sample of 48,764,719 rows from 'fame_yearly':


,registered_number,year,consolidated,turnover,shareholders_funds,profit_loss_pretax,employees,tangibles,tangibles_land_and_buildings,tangibles_land_freehold,tangibles_land_leasehold,tangibles_fixt_fit,tangibles_plant_and_vehicles,tangibles_plant,tangibles_vehicles,fixed_other,intangibles,investments_other,fixed_total,liabilities,total_assets,liabilites_lt,cos,admin_expenses,interest_paid,profit_loss_pretax2,tax,dividends,depreciation,r_and_d,remuneration_employees,wages,social_security_costs,pensions_costs,other_staff_costs,renumeration_directors,ebitda
0,10995939,2022,False,18.006,0.001,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001,0.001,NaN,0.001,NaN,NaN,-15.45,NaN,NaN,NaN,NaN,NaN,NaN,15.45,NaN,NaN,NaN,NaN,NaN,NaN
1,12904978,2023,False,NaN,1.040,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.040,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,05065103,2012,False,NaN,-30.592,NaN,NaN,165.056,NaN,NaN,NaN,NaN,NaN,NaN,NaN,165.056,NaN,NaN,165.056,-578.643,1421.676,-873.625,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,03596190,2018,False,NaN,6.659,NaN,1.0,1.868,NaN,NaN,NaN,NaN,1.868,1.868,NaN,NaN,NaN,NaN,1.868,-7.726,14.385,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.622,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07248484,2015,False,NaN,1.063,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.063,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48729,05348525,2011,False,NaN,2.627,NaN,NaN,2.386,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.386,NaN,NaN,2.386,-16.815,19.442,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48730,01106974,2013,False,NaN,0.907,NaN,NaN,38.373,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.373,15.0,NaN,53.373,-118.567,125.732,-6.258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48731,01393770,2007,False,NaN,-26.704,NaN,NaN,19.345,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.345,-95.379,69.822,-1.147,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48732,05514586,2016,False,NaN,-68.045,NaN,NaN,12.192,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.192,29.8,NaN,41.992,-12.359,43.964,-99.650,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Specific Lookup

### Firm-specific (Search by Name or ID)
Find all fixed and derived attributes for a specific company using partial string matching (case-insensitive).

In [6]:
search_term = "TESCO"  # Can be a partial name or a Registered Number
table_fixed = con.table("fame_fixed")

# Filter where company_name contains the search term OR exactly matches registered_number
search_query = table_fixed.filter(
    table_fixed["company_name"].upper().contains(search_term.upper()) |
    (table_fixed["registered_number"] == search_term)
)

# Pull the top 5 matches
df_search_results = search_query.head(10).execute()

print(f"🔍 Top search results for '{search_term}':")
display(df_search_results[["registered_number", "company_name", "ro_address", "primary_trading_address"]])

🔍 Top search results for 'TESCO':


,registered_number,company_name,ro_address,primary_trading_address
0,12302279,INTESCOM LTD,"34 New House, 67-68 Hatton Garden, London, EC1...",NaN
1,05830995,SKATESCOOL LIMITED,"15 Marroway, Weston Turville, Aylesbury, Bucki...","15 Marroway, Weston Turville, Aylesbury, Bucki..."
2,06967289,TESCO UNDERWRITING LIMITED,"The Omnibus Building, Lesbourne Road, Reigate,...",NaN
3,06003554,TESCO MAINTENANCE LIMITED,"Tesco House Shire Park, Shires Park, Kestrel W...","Tesco House Shire Park, Shires Park, Kestrel W..."
4,04345023,TESCO FREETIME LIMITED,"Tesco House Shire Park, Shires Park, Kestrel W...","Tesco House Shire Park, Shires Park, Kestrel W..."
5,03176368,TESCO INTERNATIONAL SERVICES LIMITED,"Tesco House Shire Park, Shires Park, Kestrel W...",NaN
6,05190973,MONTESCOLA LIMITED,"The Tall House 29A West Street, Marlow, Buckin...",NaN
7,07502096,TESCO FOOD SOURCING LIMITED,"Tesco House Shire Park, Shires Park, Kestrel W...","Tesco House Shire Park, Shires Park, Kestrel W..."
8,05888922,TESCO PROPERTY HOLDINGS (NO.2) LIMITED,"Tesco House Shire Park, Kestrel Way, Welwyn Ga...","Tesco House Shire Park, Kestrel Way, Welwyn Ga..."
9,05888947,TESCO FUCHSIA (FINCO2) LIMITED,"Tesco House, Delamare Road, Cheshunt, Waltham ...","Tesco House, Delamare Road, Cheshunt, Waltham ..."


### Industry-specific (by 2-digit SIC or 6-digit
- 6-digit using primary_uk_sic_2007_code in fame_fixed

In [7]:
# ### Industry-specific (by 2-digit SIC or 6-digit
# - 6-digit using primary_uk_sic_2007_code in fame_fixed
import json
import ibis
import pandas as pd

# Read build/input/SIC_priorities.json
search_inds = []

get_from_json = False
if get_from_json:
    with open(dirs.input_dir / "SIC_priorities.json", "r") as f:
        sic_priorities = json.load(f)
        search_inds = [entry.get("Division") for entry in sic_priorities if entry.get("Errored") == True]
else:
    search_inds = ["70", "74"]

# Ensure strings are properly formatted (e.g. padded with zeros) just in case
search_inds_str = [str(x).zfill(2) for x in search_inds if x is not None]
print(f"Searching for following SIC divisions individually: {search_inds_str}")

# Connect to the tables in the database
t_fixed = con.table("fame_fixed")
t_derived = con.table("fame_derived")
t_yearly = con.table("fame_yearly")

# List to accumulate our row records
results = []

for ind in search_inds_str:
    # 1. Filter the fame_derived table for the specific SIC division
    regex_pattern = rf'\b{ind}\b'
    filtered_derived = t_derived.filter(
        t_derived.industry_codes.re_search(regex_pattern)
    )

    # 2. Execute the count for fame_derived
    derived_count = filtered_derived.count().execute()

    # 3. Filter fame_fixed using a semi-join
    fixed_count = t_fixed.semi_join(filtered_derived, "registered_number").count().execute()

    # 4. Filter fame_yearly using a semi-join
    yearly_count = t_yearly.semi_join(filtered_derived, "registered_number").count().execute()

    # Append the counts for this specific division to our results list
    results.append({
        "SIC_Division": ind,
        "fame_derived": derived_count,
        "fame_fixed": fixed_count,
        "fame_yearly": yearly_count
    })

# Output the results as a formatted table
output_table = pd.DataFrame(results)

print("📊 Matching Rows by Table for Each SIC Division:")
display(output_table) # use print(output_table) if you aren't in a Jupyter environment

Searching for following SIC divisions individually: ['70', '74']
📊 Matching Rows by Table for Each SIC Division:


,SIC_Division,fame_derived,fame_fixed,fame_yearly
0,70,596707,782160,3461594
1,74,471512,560063,1820017


## 4. Time-Series Construction (Joining Fixed & Yearly)
Combine the static company metadata with its longitudinal financial performance. This demonstrates the relational integrity of the `registered_number` primary key.

In [8]:
# Pick a specific company ID to track over time
selected_ids = df_sample["registered_number"].head().tolist()

t_fixed = con.table("fame_fixed")
t_yearly = con.table("fame_yearly")

# 1. Filter both tables to the target ID
firm_fixed = t_fixed.filter(t_fixed["registered_number"].isin(selected_ids))
firm_yearly = t_yearly.filter(t_yearly["registered_number"].isin(selected_ids))

# 2. Left join yearly financials onto the fixed metadata
firm_history = firm_yearly.left_join(firm_fixed, "registered_number")

# 3. Select a curated subset of columns to display
comprehensive_view = firm_history.select(
    "registered_number",
    "company_name",
    "year",
    # Safely select financial columns if they exist in the DB
    s.contains("turnover"),
    s.contains("profit_loss_pretax"),
    s.contains("employees")
).order_by("year") # Order chronologically

display(comprehensive_view.execute())

,registered_number,company_name,year,turnover,profit_loss_pretax,profit_loss_pretax2,employees,remuneration_employees
0,05065103,BODEAN'S LIMITED,2006,NaN,NaN,NaN,NaN,NaN
1,03596190,ARISE CONSULTANTS LIMITED,2006,103.389,32.855,32.855,NaN,NaN
2,03596190,ARISE CONSULTANTS LIMITED,2007,3.525,-0.635,-0.635,NaN,NaN
3,05065103,BODEAN'S LIMITED,2007,NaN,NaN,NaN,NaN,NaN
4,03596190,ARISE CONSULTANTS LIMITED,2008,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
73,10995939,EAST &INTERNATIONAL LTD,2023,60.877,NaN,NaN,1.0,25.939
74,07248484,SPEAK TO ME FOUNDATION,2023,NaN,NaN,NaN,NaN,NaN
75,03596190,ARISE CONSULTANTS LIMITED,2024,NaN,NaN,NaN,2.0,NaN
76,12904978,ICERICASTER LTD,2024,NaN,NaN,NaN,1.0,NaN


## 5. Summary Statistics & Aggregations
Generate high-level analytical cuts (e.g., counting the number of records per year, or assessing data coverage).

In [9]:
t_yearly = con.table("fame_yearly")

# Aggregate the number of financial records available per year
# Print numbers of available records for every financial column
yearly_distribution = (
    t_yearly
    .group_by("year")
    .aggregate(
        turnover=t_yearly["turnover"].count(),
        pnl=t_yearly["profit_loss_pretax"].count(),
        employees=t_yearly["employees"].count(),
        tangibles=t_yearly["tangibles"].count(),
        t_lab=t_yearly["tangibles_land_and_buildings"].count(),
        t_land_free=t_yearly["tangibles_land_freehold"].count(),
        t_land_lease=t_yearly["tangibles_land_leasehold"].count(),
        fixed_other=t_yearly["fixed_other"].count(),
        intangibles=t_yearly["intangibles"].count(),
        fixed_total=t_yearly["fixed_total"].count(),
        liabilities=t_yearly["liabilities"].count(),
        t_assets=t_yearly["total_assets"].count(),
        liabilites_lt=t_yearly["liabilites_lt"].count(),
        cos=t_yearly["cos"].count(),
        dividends=t_yearly["dividends"].count(),
        r_and_d=t_yearly["r_and_d"].count(),
        renum=t_yearly["remuneration_employees"].count(),
        wages=t_yearly["wages"].count(),
        ss_cost=t_yearly["social_security_costs"].count(),
        pension_cost=t_yearly["pensions_costs"].count(),
        ebitda=t_yearly["ebitda"].count(),
        all=t_yearly.count()
    )
    .order_by(ibis.desc("year"))
)

print("📈 Data coverage by year:")
# Format table with commas for readability and display
count_table: pd.DataFrame = yearly_distribution.execute()
count_table_formatted = count_table.copy().reset_index(drop=True)
for col in count_table_formatted.columns:
    if col != "year":
        count_table_formatted[col] = count_table_formatted[col].apply(lambda x: f"{x:,}")
display(count_table_formatted)

📈 Data coverage by year:


,year,turnover,pnl,employees,tangibles,t_lab,t_land_free,t_land_lease,fixed_other,intangibles,fixed_total,liabilities,t_assets,liabilites_lt,cos,dividends,r_and_d,renum,wages,ss_cost,pension_cost,ebitda,all
0,2025,807,910,"8,996","7,937",452,323,146,"5,747",489,"11,244","12,480","18,599","6,083",1,0,0,0,0,0,0,2,"24,459"
1,2024,"94,494","105,784","1,340,734","1,047,073","105,418","85,615","24,183","719,463","63,027","1,264,069","1,533,438","1,894,533","815,463","38,851","13,574","1,484","59,267","44,859","36,012","36,738","108,787","2,092,655"
2,2023,"225,061","261,705","2,446,895","2,040,219","239,927","187,984","63,139","1,372,499","150,949","2,481,258","3,011,751","3,691,174","1,643,036","105,778","35,630","5,833","142,480","118,546","97,413","95,691","258,139","3,820,456"
3,2022,"225,468","262,483","2,394,142","1,997,487","236,652","184,034","64,157","1,346,771","151,632","2,420,948","2,946,393","3,608,343","1,621,655","105,398","34,317","5,866","141,410","117,354","96,589","93,971","258,813","3,731,057"
4,2021,"223,771","260,177","2,319,265","1,938,160","228,567","176,745","62,748","1,302,335","151,569","2,332,535","2,869,517","3,495,129","1,577,048","102,957","33,061","5,513","141,902","117,380","95,307","91,924","256,884","3,609,155"
5,2020,"224,351","258,721","2,209,136","1,854,461","225,155","174,985","61,311","1,248,282","150,641","2,206,124","2,768,068","3,322,223","1,321,586","102,433","33,665","5,376","140,437","123,172","94,986","89,910","255,436","3,429,017"
6,2019,"225,920","257,298","1,627,788","1,766,406","231,522","185,925","58,263","1,213,688","151,042","2,092,388","2,631,612","3,147,171","1,086,304","102,654","37,807","5,401","138,495","136,861","93,584","85,807","254,290","3,245,943"
7,2018,"227,966","258,633","1,357,873","1,700,258","214,419","173,421","52,653","1,109,981","153,905","2,000,357","2,503,485","2,989,221","1,025,002","105,614","41,363","4,901","142,120","141,048","94,069","84,134","265,801","3,082,598"
8,2017,"240,199","273,146","1,148,559","1,641,969","209,308","170,094","50,278","1,054,784","161,714","1,900,622","2,407,373","2,859,013","962,126","111,547","46,063","4,762","159,247","158,093","102,637","84,731","286,617","2,945,458"
9,2016,"222,772","247,317","629,484","1,573,656","99,762","74,697","33,433","1,317,533","205,522","1,824,691","2,283,605","2,708,697","833,353","110,857","51,875","3,713","174,343","172,524","121,858","88,230","254,510","2,784,094"


## 6. Exporting Queries to Excel
Any queried subset of data can be instantly dumped into an Excel file for offline review.

In [11]:
import pandas as pd
import ibis

approach = 'random' # first

# Dump a random sample of 500 rows from each table to a single .xlsx file in /tmp
row_count = 2000
out_file_raw = dirs.output_dir / "df_raw_sample.xlsx"
desired_tables = ["fame_derived", "fame_fixed", "fame_yearly_filtered", "lars_fixed", "lars_yearly"]
exists_tables = con.list_tables()
intersection = set(desired_tables).intersection(exists_tables)

# but we want to have 5 separate tabs in the same excel file, one for each table, with the table name as the tab name
con = ibis.duckdb.connect(str(dirs.output_dir / "fame_data.duckdb"))
with pd.ExcelWriter(out_file_raw, engine='openpyxl') as writer:
    # df_raw_head.to_excel(writer, sheet_name='df_raw', index=False)
    for table in intersection:
        if approach == 'first':
            con.table(table).head(row_count).execute().to_excel(writer, sheet_name=table, index=False)
        elif approach == 'random':
            nb_rows = con.table(table).count().execute()
            fraction = row_count / nb_rows if nb_rows > row_count else 1.0
            df_table_rd = con.table(table).sample(fraction, seed=12345).execute()
            df_table_rd.to_excel(writer, sheet_name=table, index=False)
        
print(f"✅ Successfully dumped a random {row_count} rows of df_raw to: {out_file_raw}")

✅ Successfully dumped a random 2000 rows of df_raw to: C:\Users\lazyst\Files\ucl\Dissertation\build\output\df_raw_sample.xlsx


# 7. Industry counts in each of the tables
### [read] Count cells belong to certain industries
- Using the fame_derived system
- Remove rows belonging to a certain industry, as a 'fresh start' mechanism

In [12]:
# connect to fame_derived and group by industry_codes
# count the number of rows belonging to each group (unique industry_codes)
# Sort the results in ascending order of industry_codes
# Output the result (as a table, with counts as the column)
import ibis
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))
dirs = get_data_dirs()
fame_derived = con.table("fame_derived")
industry_counts = (
    fame_derived
    .group_by("industry_codes")
    .aggregate(count=fame_derived.count())
    .order_by("industry_codes")
)
# Execute and display the results
df_industry_counts = industry_counts.execute()
print("📊 Industry Codes and Their Counts:")
for index, row in df_industry_counts.iterrows():
    print(f"Industry Code: {row['industry_codes']}, Count: {row['count']:,}")

📊 Industry Codes and Their Counts:
Industry Code: 01, Count: 42,142
Industry Code: 02, Count: 7,744
Industry Code: 03, Count: 6,573
Industry Code: 05, Count: 444
Industry Code: 06, Count: 4,445
Industry Code: 07, Count: 1,036
Industry Code: 08, Count: 3,748
Industry Code: 09, Count: 11,152
Industry Code: 10, Count: 33,141
Industry Code: 11, Count: 9,878
Industry Code: 12, Count: 294
Industry Code: 13, Count: 13,800
Industry Code: 14, Count: 19,895
Industry Code: 15, Count: 3,526
Industry Code: 16, Count: 16,828
Industry Code: 17, Count: 6,168
Industry Code: 18, Count: 31,021
Industry Code: 19, Count: 781
Industry Code: 20, Count: 11,451
Industry Code: 21, Count: 3,259
Industry Code: 22, Count: 12,727
Industry Code: 23, Count: 9,020
Industry Code: 24, Count: 6,164
Industry Code: 25, Count: 49,205
Industry Code: 26, Count: 19,531
Industry Code: 27, Count: 10,650
Industry Code: 28, Count: 20,683
Industry Code: 29, Count: 9,243
Industry Code: 30, Count: 9,196
Industry Code: 31, Count: 18,1